In [1]:
import sys
from pathlib import Path

# Make project root (the folder that contains `src`) importable
project_root = Path().resolve().parent  # this should be R-learner-LTE
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import logging
from typing import Tuple

import hydra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import DictConfig, OmegaConf

from src.data.base_dataset import TwoSampleDataSplit, GroundTruth
from src.data.semi_synthetic import IST3SemiSyntheticDataset
from src.data.utils import split_two_sample_data
from src.model.t_learner import TLearner
from src.model.t_r_learner import tRlearner

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
def load_config(
    dataset: str | None = None,
    model: str | None = None,
    trainer: str | None = None,
    config_root: Path = Path("config"),
):
    """
    Load full config, optionally overriding dataset/model/trainer from config.yaml defaults.
    """
    base_cfg = OmegaConf.load(config_root / "config.yaml")
    defaults = base_cfg.get("defaults", [])

    def _get_default(name: str) -> str | None:
        for item in defaults:
            if isinstance(item, dict) and name in item:
                return item[name]
        return None

    dataset_name = dataset or _get_default("dataset")
    model_name = model or _get_default("model")
    trainer_name = trainer or _get_default("trainer")

    cfg = OmegaConf.create()
    cfg.dataset = OmegaConf.load(config_root / "dataset" / f"{dataset_name}.yaml")
    cfg.model = OmegaConf.load(config_root / "model" / f"{model_name}.yaml")
    cfg.trainer = OmegaConf.load(config_root / "trainer" / f"{trainer_name}.yaml")

    # copy other top-level fields (e.g. seed)
    for key, value in base_cfg.items():
        if key != "defaults":
            cfg[key] = value

    return cfg

In [3]:
config_root = 'C:\\Users\\ma\\Research\\Long-term-effects\\R-learner-LTE\\config'
cfg = load_config(dataset="semi-synthetic", model="R-learner", trainer="default", config_root=Path(config_root))

In [4]:
ist = IST3SemiSyntheticDataset(cfg.dataset)
data, gt = ist.sample()

INFO:src.data.semi_synthetic:Loaded real data from C:\Users\ma\Research\Long-term-effects\R-learner-LTE\src\data\semi_synthetic_real_data.csv with shape (2720, 32)
INFO:src.data.semi_synthetic:Numerical Columns: ['age', 'randdelay', 'sbprand', 'dbprand', 'weight', 'glucose', 'gcs_score_rand', 'nihss', 'R_infarct_size', 'R_hypodensity', 'R_swelling']
INFO:src.data.semi_synthetic:Categorical Columns: ['gender', 'country', 'livealone_rand', 'indepinadl_rand', 'atrialfib_rand', 'stroke_pre', 'hypertension_pre', 'diabetes_pre', 'aspirin_pre', 'other_antiplat_pre', 'anticoag_pre', 'stroketype']
INFO:root:Semi-synthetic dataset: n_e = 636, n_o = 2084


In [ ]:
#
def compute_variance_pseudo_outcome(X: np.ndarray, ist_dataset: IST3SemiSyntheticDataset, sigma_y = 1, sigma_h = 0.5) -> np.ndarray:
    """
    \begin{align}
        \var(\phi_{DR} \mid X) \leq \underbrace{(1-\rho(X)) \cdot \Sigma_{t}(X)}_{\text{Treatment Overlap Instability}} + \underbrace{\rho(X) \cdot \Sigma_{\text{o}}(X)}_{\text{Observation Overlap Instability}}
    \end{align}
    where:
    \begin{align}
        \Sigma_{t}(X) &= \mathbb{E}\left[ \left(\frac{1}{\pi(X)(1-\pi(X))}\right)^2 \var(h(S,X) \mid X, A, R=0) \bigg| X \right] \notag \\
        \Sigma_{\text{o}}(X) &= \mathbb{E}_{S|X} \left[ \left(\frac{1-\rho(S,X)}{\rho(S,X)}\right)^2 \var(Y \mid S, X, R=1) \right]
    \end{align}
    """
    #\frac{1}{\pi(X)(1-\pi(X))}\right)^2 \var(h(S,X)
    pi = ist_dataset.pi_E(X)
    sigma_t = np.sqrt((1 / (pi * (1 - pi))) ** 2 * sigma_h ** 2)
    sigma_o = np.sqrt(((1 - ist_dataset.rho_O(X)) / ist_dataset.rho_O(X)) ** 2 * sigma_y ** 2)
    var_pseudo_outcome = (1 - ist_dataset.rho_O(X)) * sigma_t + ist_dataset.rho_O(X) * sigma_o

    t_overlap = pi * (1 - pi)
    o_overlap = ist_dataset.rho_O(X)


    return var_pseudo_outcome

TypeError: 'TwoSampleDataSplit' object is not subscriptable